# Text extraction pipeline

Saved split → raw GPU OCR → filter fragments → join remaining fragments → save `candidates.jsonl`.

Filtering happens **before joining**, without labels:
- Reject whole fragments containing non-English characters, including mixed English/Japanese fragments. Standard English quotes, dashes, and ellipses are allowed.
- Remove listed watermarks, sound words, repeated listed sounds, and explicit `SFX:` captions.
- Remove small standalone numbers near the top/bottom page margins. Numbers in dialogue and larger central number boxes are preserved.
- Remove blank, invalid, low-score, and duplicate detections.

Edit sound words in `datasets/rules/common_sound_effects.json`. Rules are heuristics and do not classify image content. GPU extraction settings remain unchanged. Cached raw OCR is reused when available.


In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = None
for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    for candidate in (base, base / "manga_notebook_codebase"):
        if (candidate / "src/text_extraction_pipeline.py").is_file():
            ROOT = candidate
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise RuntimeError("Start this notebook inside the project or workspace")
sys.path.insert(0, str(ROOT / "src"))
# Reload our helpers when rerunning after a filtering-code change.
import importlib
import text_extraction_pipeline
importlib.reload(text_extraction_pipeline)
from text_extraction_pipeline import select_rows, run_pipeline, load_sound_words
from data_utils import load_jsonl
print("Project:", ROOT)


Project: C:\Users\ESHAAN\BYTE\Applied-Task\MultiModelSystem\manga_notebook_codebase


In [7]:
# Change these values, then rerun the following cells.
SPLIT = "val"                   # train / val / test / all
INPUT_ROOT = None               # e.g. Path("C:/data/new_manga"); images/<id>/01.png,02.png,03.png
LIMIT = None                    # None = entire selected split; 1 = one triplet
MIN_OCR_SCORE = 0.0             # Keep all valid confidence levels by default
MAX_JOIN_GAP = 1.25
CENTER_TOLERANCE = 1.5
SOUND_EFFECTS_PATH = ROOT / "datasets/rules/common_sound_effects.json"
RUN_NAME = SPLIT if INPUT_ROOT is None else "unseen"
OUTPUT_DIR = ROOT / "outputs/text_extraction" / RUN_NAME
RAW_CACHE = (ROOT / "outputs/ocr/paddleocr/candidates.jsonl"
             if INPUT_ROOT is None else OUTPUT_DIR / "raw_candidates.jsonl")
OCR_PYTHON = Path(os.environ.get("MANGA_OCR_PYTHON", str(ROOT.parent / ".venvOCR/Scripts/python.exe")))


## Load the split and inspect inputs
The saved split is loaded directly from `outputs/split.json`. Selection uses sequence IDs and ordered image paths; training targets and reference transcripts never enter inference. `INPUT_ROOT` bypasses the saved split for unseen image folders.


In [8]:
source, selected_rows = select_rows(ROOT, SPLIT, INPUT_ROOT, LIMIT)
print(f"Selected {len(selected_rows)} sequences / {len(selected_rows) * 3} pages")
print("First sequence:", selected_rows[0]["sequence_id"])
print("Images:", [str(p) for p in selected_rows[0]["images"]])
sound_words = load_sound_words(SOUND_EFFECTS_PATH)
print(f"Sound-effect list ({len(sound_words)} words):", ", ".join(sorted(sound_words)))
print("Final output:", OUTPUT_DIR / "candidates.jsonl")


Selected 16 sequences / 48 pages
First sequence: seq_b8b5563330fdc89d
Images: ['C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\MultiModelSystem\\manga_notebook_codebase\\TaskData\\development\\images\\seq_b8b5563330fdc89d\\01.png', 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\MultiModelSystem\\manga_notebook_codebase\\TaskData\\development\\images\\seq_b8b5563330fdc89d\\02.png', 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\MultiModelSystem\\manga_notebook_codebase\\TaskData\\development\\images\\seq_b8b5563330fdc89d\\03.png']
Sound-effect list (69 words): BAM, BANG, BOOM, BUZZ, BZZT, CHAK, CHOMP, CLACK, CLANG, CLICK, CLUNK, COM, CRACK, CRASH, CREAK, DING, DO NOT MIRROR, DOINK, DONOTMIRROR, DRIP, FSHH, FWIP, GLUB, GLUB GLUB, GULP, K-CHAK, KABOOM, KLIK, KLINK, KNOUK, KRAKDOW, KRAKKRAKKRAK, KREAK, KREEK, KREPT, KRICH, KRIK, MANGA, PING, PINGGGG, PLIP, POW, RUSTL, RUSTLE, SFX: GULP, SFX: HOOFBEATS, SHIK, SHOOP, SKRSH, SLAM, SMASH, SNAP, SPLASH, STREAM.COM, SWISH, SWOOSH, TAP, TAPT, THUD, THUNK, TICK, TOCK, 

## Run extraction, filtering, and joining
Existing raw candidates remain unchanged. Missing sequences are extracted into this run's `raw_candidates.jsonl`. Sound words are removed **before** geometry-based joining, so they cannot become part of neighboring dialogue groups.

Final candidates contain `id`, `text`, pixel `box`, mean OCR `score`, and page-local source `candidate_ids`. The three page lists remain in image order. Group ordering is spatial; this stage does not solve manga panel reading order or assign speakers.


In [9]:
records, summary = run_pipeline(
    ROOT, split=SPLIT, input_root=INPUT_ROOT, raw_cache=RAW_CACHE,
    output_dir=OUTPUT_DIR, ocr_python=OCR_PYTHON, limit=LIMIT,
    min_score=MIN_OCR_SCORE, max_gap=MAX_JOIN_GAP,
    center_tolerance=CENTER_TOLERANCE, sound_effects_path=SOUND_EFFECTS_PATH,
)


{
  "sequences": 16,
  "pages": 48,
  "raw_candidates": 1456,
  "excluded_candidates": 99,
  "joined_candidates": 392,
  "output": "C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\MultiModelSystem\\manga_notebook_codebase\\outputs\\text_extraction\\val\\candidates.jsonl"
}


## Verify and inspect the saved output
`filter_audit.jsonl` records every discarded fragment and its reason. `candidates.metadata.json` records the split, settings, blacklist, code, image and input/output hashes. Reused legacy OCR is explicitly marked as lacking its original runtime/image provenance; regenerate raw OCR into a fresh cache for a fully fingerprinted run.


In [10]:
from collections import Counter

final_path = OUTPUT_DIR / "candidates.jsonl"
saved = load_jsonl(final_path)
assert [r["sequence_id"] for r in saved] == [r["sequence_id"] for r in selected_rows]
assert all(len(r["pages"]) == 3 for r in saved)

# Each original fragment should appear in at most one joined group.
for record in saved:
    for page in record["pages"]:
        source_ids = [cid for group in page for cid in group["candidate_ids"]]
        assert len(source_ids) == len(set(source_ids))

# Show removals by rule, then a few examples of each.
audit = load_jsonl(OUTPUT_DIR / "filter_audit.jsonl")
removed = [entry for page in audit for entry in page["excluded"]]
counts = Counter(entry["reason"] for entry in removed)
print("Removed fragments by rule:", dict(counts))
for reason in counts:
    examples = [entry["candidate"]["text"] for entry in removed if entry["reason"] == reason]
    print(reason, "examples:", examples[:8])
print("Saved:", final_path)


Removed fragments by rule: {'non_english_fragment': 20, 'page_number': 14, 'sound_effect_word': 21, 'watermark_phrase': 41, 'blank_text': 3}
non_english_fragment examples: ['月', '月', '月', 'い', 'あん', 'めん', '魚定食', '魚定食']
page_number examples: ['8', '9', '10', '42', '43', '44', '7', '80']
sound_effect_word examples: ['SKRSH', 'SKRSH', 'THUNK', 'THUD', 'SHOOP', 'FWIP', 'BAM', 'SNAP']
watermark_phrase examples: ['MANGA', 'DO NOT MIRROR', 'MANGA', 'STREAM.com', 'DONOTMIRROR', 'MANGA', 'DO NOT MIRROR', 'MANGA']
blank_text examples: ['', '', '']
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\MultiModelSystem\manga_notebook_codebase\outputs\text_extraction\val\candidates.jsonl


## Next step: validation diagnostics
After a validation run, open [validation_ocr_joining.ipynb](ocr_text_extraction/validation_ocr_joining.ipynb).
It evaluates this notebook's saved `outputs/text_extraction/val/candidates.jsonl`, generates reference comparisons and page previews, and lists upstream exclusions. It does not rerun OCR, filtering, or joining.
